# 04. Usage Feature Engineering

이 노트북의 목적은 `02_preprocessing_policy.ipynb`에서 확정한 고객별 3주 관측창 데이터를 사용해, 구독 이벤트 단위의 시청 행동 파생변수를 생성하는 것이다.

이 단계는 영화 장르나 콘텐츠 메타데이터를 사용하지 않는다. 콘텐츠 메타데이터 기반 피처는 `05_content_feature_engineering.ipynb`에서 별도로 만든다.

핵심 원칙은 다음과 같다.

1. 분석 단위는 `membership_row_id`이다.
2. 시청 행동은 고객별 `reg_date` 기준 day 0~20 관측창 안의 기록만 사용한다.
3. 4주차 day 21~27은 리텐션 대응기간이므로 피처로 사용하지 않는다.
4. 시청이력이 없는 고객은 제거하지 않고, `no_watch_obs_flag`로 보존한다.
5. 단순 시청량뿐 아니라 초반 루틴화, 후반 몰아보기, 시청 공백, 주차별 변화량을 함께 만든다.

주요 출력 파일은 다음과 같다.

- `_data/02_interim/user_usage_features.csv`
- `_data/03_processed/modeling_feature_table_usage.csv`
- `_data/02_interim/usage_feature_summary.json`
- `reports/tables/04_usage_feature_*.csv`

## 4-1. 라이브러리 로딩

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

## 4-2. 경로 설정

이 노트북은 `park.ingyeom` 폴더 안에서 실행하는 것을 기준으로 한다. 02번 노트북이 먼저 실행되어 `_data/02_interim`에 전처리 산출물이 저장되어 있어야 한다.

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / '_data').exists() and (p / 'notebooks').exists():
            return p
        if p.name == 'park.ingyeom':
            return p
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / '_data'
RAW_DIR = DATA_DIR / '01_raw'
INTERIM_DIR = DATA_DIR / '02_interim'
PROCESSED_DIR = DATA_DIR / '03_processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
TABLES_DIR = REPORTS_DIR / 'tables'

for d in [INTERIM_DIR, PROCESSED_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('INTERIM_DIR:', INTERIM_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('TABLES_DIR:', TABLES_DIR)

## 4-3. 02번 산출물 로딩

이 노트북의 입력은 원본 CSV가 아니라 02번 전처리 노트북의 산출물이다.

In [ ]:
INPUT_FILES = {
    'membership_preprocessed': INTERIM_DIR / 'membership_preprocessed.csv',
    'membership_with_usernum': INTERIM_DIR / 'membership_with_usernum.csv',
    'obs_view': INTERIM_DIR / 'view_history_observation_window.csv',
}

missing = [str(p) for p in INPUT_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        '02_preprocessing_policy.ipynb 산출물이 없습니다. 먼저 02번 노트북을 실행하세요. Missing: ' + str(missing)
    )

membership = pd.read_csv(INPUT_FILES['membership_preprocessed'])
membership_user = pd.read_csv(INPUT_FILES['membership_with_usernum'])
obs_view = pd.read_csv(INPUT_FILES['obs_view'])

for col in ['reg_date', 'end_date']:
    if col in membership.columns:
        membership[col] = pd.to_datetime(membership[col])
    if col in membership_user.columns:
        membership_user[col] = pd.to_datetime(membership_user[col])

if 'watch_day' in obs_view.columns:
    obs_view['watch_day'] = pd.to_datetime(obs_view['watch_day'])

if 'watch_time(min)' in obs_view.columns and 'watch_time' not in obs_view.columns:
    obs_view = obs_view.rename(columns={'watch_time(min)': 'watch_time'})

file_summary = pd.DataFrame([
    {'name': 'membership_preprocessed', 'path': str(INPUT_FILES['membership_preprocessed']), 'rows': len(membership), 'cols': membership.shape[1]},
    {'name': 'membership_with_usernum', 'path': str(INPUT_FILES['membership_with_usernum']), 'rows': len(membership_user), 'cols': membership_user.shape[1]},
    {'name': 'view_history_observation_window', 'path': str(INPUT_FILES['obs_view']), 'rows': len(obs_view), 'cols': obs_view.shape[1]},
])

file_summary.to_csv(TABLES_DIR / '04_usage_feature_input_file_summary.csv', index=False, encoding='utf-8-sig')
display(file_summary)

## 4-4. 입력 데이터 검산

04번은 시청 행동 피처를 만드는 단계이므로, 02번에서 정의한 관측창이 유지되어 있는지 먼저 확인한다.

In [ ]:
required_membership_cols = {'membership_row_id', 'USER_KEY', 'is_repurchase', 'is_100won', 'max_screen', 'has_watch_obs', 'no_watch_obs_flag'}
required_obs_cols = {'membership_row_id', 'USER_NUM', 'MOVIE_NUM', 'watch_day', 'watch_time', 'watch_rel_day', 'obs_week'}

missing_membership_cols = sorted(required_membership_cols - set(membership.columns))
missing_obs_cols = sorted(required_obs_cols - set(obs_view.columns))

if missing_membership_cols:
    raise ValueError(f'membership_preprocessed 필수 컬럼 누락: {missing_membership_cols}')
if missing_obs_cols:
    raise ValueError(f'view_history_observation_window 필수 컬럼 누락: {missing_obs_cols}')

obs_check = pd.DataFrame([
    {'item': 'membership_rows', 'value': len(membership)},
    {'item': 'unique_membership_row_id_membership', 'value': membership['membership_row_id'].nunique()},
    {'item': 'obs_view_rows', 'value': len(obs_view)},
    {'item': 'unique_membership_row_id_obs_view', 'value': obs_view['membership_row_id'].nunique()},
    {'item': 'obs_min_watch_rel_day', 'value': obs_view['watch_rel_day'].min()},
    {'item': 'obs_max_watch_rel_day', 'value': obs_view['watch_rel_day'].max()},
    {'item': 'obs_unique_movies', 'value': obs_view['MOVIE_NUM'].nunique()},
    {'item': 'obs_watch_time_sum', 'value': obs_view['watch_time'].sum()},
])

obs_check.to_csv(TABLES_DIR / '04_usage_feature_observation_input_check.csv', index=False, encoding='utf-8-sig')
display(obs_check)

## 4-5. 기본 시청량 피처 생성

기본 시청량 피처는 고객별 총 시청시간, 세션 수, 고유 콘텐츠 수, 고유 시청일 수를 포함한다. 단순 시청량은 이전 분석에서 강한 신호가 아니었지만, 모델링과 검정의 기본 피처로 유지한다.

In [ ]:
def build_basic_usage_features(obs: pd.DataFrame) -> pd.DataFrame:
    basic = obs.groupby('membership_row_id').agg(
        total_watch_time=('watch_time', 'sum'),
        total_sessions=('watch_time', 'count'),
        unique_contents=('MOVIE_NUM', 'nunique'),
        unique_days=('watch_rel_day', 'nunique'),
        first_watch_rel_day=('watch_rel_day', 'min'),
        last_watch_rel_day=('watch_rel_day', 'max'),
        avg_session_time=('watch_time', 'mean'),
        median_session_time=('watch_time', 'median'),
        min_session_time=('watch_time', 'min'),
        max_session_time=('watch_time', 'max'),
        std_session_time=('watch_time', 'std'),
    ).reset_index()

    basic['std_session_time'] = basic['std_session_time'].fillna(0)
    basic['active_span_days'] = (basic['last_watch_rel_day'] - basic['first_watch_rel_day'] + 1).clip(lower=0)
    basic['days_since_first_watch_from_reg'] = basic['first_watch_rel_day']
    basic['days_since_last_watch_to_obs_end'] = 20 - basic['last_watch_rel_day']
    basic['watch_days_ratio'] = basic['unique_days'] / 21

    basic['sessions_per_active_day'] = np.where(
        basic['unique_days'] > 0,
        basic['total_sessions'] / basic['unique_days'],
        0,
    )
    basic['contents_per_active_day'] = np.where(
        basic['unique_days'] > 0,
        basic['unique_contents'] / basic['unique_days'],
        0,
    )
    basic['avg_watch_time_per_content'] = np.where(
        basic['unique_contents'] > 0,
        basic['total_watch_time'] / basic['unique_contents'],
        0,
    )
    return basic

basic_usage = build_basic_usage_features(obs_view)
display(basic_usage.head())

## 4-6. 일별 시청량과 몰아보기 피처

몰아보기 피처는 특정 하루에 시청시간이 몰리는지를 확인하기 위한 변수다. 총 시청시간 자체보다 시청이 특정 시점에 집중되는지가 이탈 행동과 더 관련될 수 있다.

In [ ]:
def build_daily_usage_features(obs: pd.DataFrame) -> pd.DataFrame:
    daily = obs.groupby(['membership_row_id', 'watch_rel_day'], as_index=False).agg(
        daily_watch_time=('watch_time', 'sum'),
        daily_sessions=('watch_time', 'count'),
        daily_unique_contents=('MOVIE_NUM', 'nunique'),
    )

    daily_agg = daily.groupby('membership_row_id').agg(
        max_daily_watch_time=('daily_watch_time', 'max'),
        avg_daily_watch_time=('daily_watch_time', 'mean'),
        median_daily_watch_time=('daily_watch_time', 'median'),
        std_daily_watch_time=('daily_watch_time', 'std'),
        max_daily_sessions=('daily_sessions', 'max'),
    ).reset_index()

    daily_agg['std_daily_watch_time'] = daily_agg['std_daily_watch_time'].fillna(0)

    return daily, daily_agg

daily_usage_long, daily_usage = build_daily_usage_features(obs_view)
display(daily_usage.head())

## 4-7. 시청 간격과 공백 피처

OTT 이탈은 총량보다 습관성의 문제일 수 있다. 따라서 시청일 사이의 간격과 관측창 내 최대 비활성 구간을 계산한다.

In [ ]:
def compute_gap_features(day_values: pd.Series) -> pd.Series:
    days = sorted(pd.Series(day_values).dropna().astype(int).unique())
    if len(days) == 0:
        return pd.Series({
            'avg_gap_between_watch_days': np.nan,
            'max_gap_between_watch_days': np.nan,
            'max_inactive_gap_days': 21,
            'start_inactive_days': 21,
            'end_inactive_days': 21,
        })

    start_inactive = days[0]
    end_inactive = 20 - days[-1]

    if len(days) >= 2:
        diffs = np.diff(days)
        avg_gap = float(np.mean(diffs))
        max_gap = float(np.max(diffs))
        internal_inactive = int(max(np.max(diffs) - 1, 0))
    else:
        avg_gap = np.nan
        max_gap = np.nan
        internal_inactive = 0

    max_inactive = max(int(start_inactive), int(end_inactive), int(internal_inactive))

    return pd.Series({
        'avg_gap_between_watch_days': avg_gap,
        'max_gap_between_watch_days': max_gap,
        'max_inactive_gap_days': max_inactive,
        'start_inactive_days': int(start_inactive),
        'end_inactive_days': int(end_inactive),
    })

watch_gap_features = (
    obs_view.groupby('membership_row_id')['watch_rel_day']
    .apply(compute_gap_features)
    .unstack()
    .reset_index()
)

watch_gap_features['avg_gap_between_watch_days'] = watch_gap_features['avg_gap_between_watch_days'].fillna(0)
watch_gap_features['max_gap_between_watch_days'] = watch_gap_features['max_gap_between_watch_days'].fillna(0)

display(watch_gap_features.head())

## 4-8. 주차별 시청 피처

관측창 day 0~20을 1주차, 2주차, 3주차로 나누어 시청시간, 세션 수, 고유 콘텐츠 수, 활성일 수를 계산한다.

In [ ]:
def pivot_weekly(obs: pd.DataFrame, value_col: str, aggfunc, prefix: str) -> pd.DataFrame:
    table = (
        obs.pivot_table(
            index='membership_row_id',
            columns='obs_week',
            values=value_col,
            aggfunc=aggfunc,
            fill_value=0,
        )
        .reset_index()
    )
    rename = {w: f'week{int(w)}_{prefix}' for w in [1, 2, 3] if w in table.columns}
    table = table.rename(columns=rename)
    for w in [1, 2, 3]:
        col = f'week{w}_{prefix}'
        if col not in table.columns:
            table[col] = 0
    return table[['membership_row_id', f'week1_{prefix}', f'week2_{prefix}', f'week3_{prefix}']]

weekly_watch = pivot_weekly(obs_view, 'watch_time', 'sum', 'watch_time')
weekly_sessions = pivot_weekly(obs_view, 'watch_time', 'count', 'sessions')
weekly_contents = pivot_weekly(obs_view, 'MOVIE_NUM', pd.Series.nunique, 'unique_contents')
weekly_days = pivot_weekly(obs_view, 'watch_rel_day', pd.Series.nunique, 'active_days')

weekly_features = (
    weekly_watch
    .merge(weekly_sessions, on='membership_row_id', how='outer')
    .merge(weekly_contents, on='membership_row_id', how='outer')
    .merge(weekly_days, on='membership_row_id', how='outer')
    .fillna(0)
)

display(weekly_features.head())

## 4-9. 초반 루틴화, 후반 몰아보기, 변화량 피처

이 단계에서 프로젝트의 핵심 행동 가설을 수치화한다. 이전 탐색에서 단순 시청량보다 1주차 비중, 3주차 증가량, 후반 몰아보기 패턴이 더 의미 있는 후보로 나타났다.

In [ ]:
def add_weekly_pattern_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ['week1_watch_time', 'week2_watch_time', 'week3_watch_time']:
        if col not in out.columns:
            out[col] = 0

    total_week_watch = out[['week1_watch_time', 'week2_watch_time', 'week3_watch_time']].sum(axis=1)

    out['week1_ratio'] = np.where(total_week_watch > 0, out['week1_watch_time'] / total_week_watch, 0)
    out['week2_ratio'] = np.where(total_week_watch > 0, out['week2_watch_time'] / total_week_watch, 0)
    out['week3_ratio'] = np.where(total_week_watch > 0, out['week3_watch_time'] / total_week_watch, 0)

    out['front_loaded_ratio'] = out['week1_ratio']
    out['late_ratio'] = out['week3_ratio']

    out['w2_minus_w1_watch_time'] = out['week2_watch_time'] - out['week1_watch_time']
    out['w3_minus_w2_watch_time'] = out['week3_watch_time'] - out['week2_watch_time']
    out['w3_minus_w1_watch_time'] = out['week3_watch_time'] - out['week1_watch_time']

    out['w3_to_w1_ratio'] = np.where(
        out['week1_watch_time'] > 0,
        out['week3_watch_time'] / out['week1_watch_time'],
        np.where(out['week3_watch_time'] > 0, np.inf, 0),
    )
    out['w3_to_w1_ratio_capped'] = out['w3_to_w1_ratio'].replace(np.inf, 999).clip(upper=999)

    # 1주차 중심점 day 3, 3주차 중심점 day 17 사이의 변화량을 간단한 slope로 둔다.
    out['daily_watch_slope'] = out['w3_minus_w1_watch_time'] / 14

    out['week_count_with_watch'] = (
        (out['week1_watch_time'] > 0).astype(int)
        + (out['week2_watch_time'] > 0).astype(int)
        + (out['week3_watch_time'] > 0).astype(int)
    )

    out['front_loaded_flag'] = (out['week1_ratio'] >= 0.5).astype(int)
    out['late_binge_flag'] = (out['week3_ratio'] >= 0.5).astype(int)
    out['only_week1_flag'] = ((out['week1_watch_time'] > 0) & (out['week2_watch_time'] == 0) & (out['week3_watch_time'] == 0)).astype(int)
    out['only_week3_flag'] = ((out['week1_watch_time'] == 0) & (out['week2_watch_time'] == 0) & (out['week3_watch_time'] > 0)).astype(int)
    out['no_week1_flag'] = (out['week1_watch_time'] == 0).astype(int)
    out['no_week3_flag'] = (out['week3_watch_time'] == 0).astype(int)
    out['steady_3week_watch_flag'] = (out['week_count_with_watch'] == 3).astype(int)

    out['usage_trend_label'] = np.select(
        [
            out['only_week1_flag'] == 1,
            out['only_week3_flag'] == 1,
            out['w3_minus_w1_watch_time'] > 30,
            out['w3_minus_w1_watch_time'] < -30,
            out['steady_3week_watch_flag'] == 1,
        ],
        [
            'only_week1',
            'only_week3',
            'late_increasing',
            'front_decreasing',
            'steady_all_weeks',
        ],
        default='mixed_or_low_activity',
    )

    return out

## 4-10. 전체 usage feature table 조립

In [ ]:
usage_features = (
    basic_usage
    .merge(daily_usage, on='membership_row_id', how='left')
    .merge(watch_gap_features, on='membership_row_id', how='left')
    .merge(weekly_features, on='membership_row_id', how='left')
)

usage_features = add_weekly_pattern_features(usage_features)

usage_features['max_day_share'] = np.where(
    usage_features['total_watch_time'] > 0,
    usage_features['max_daily_watch_time'] / usage_features['total_watch_time'],
    0,
)
usage_features['one_day_binge_flag'] = (usage_features['max_day_share'] >= 0.8).astype(int)
usage_features['has_usage_feature'] = 1

# Ensure stable column order.
front_cols = ['membership_row_id', 'has_usage_feature']
usage_features = usage_features[front_cols + [c for c in usage_features.columns if c not in front_cols]]

usage_features = usage_features.replace([np.inf, -np.inf], np.nan)

display(usage_features.head())
print('usage_features shape:', usage_features.shape)

## 4-11. 시청이력 없는 고객 포함 모델링 테이블 생성

시청이력 없는 고객은 삭제하지 않는다. 04번에서 생성한 피처를 membership table에 left join하고, 시청이력 없는 고객의 행동 피처는 0 또는 별도 플래그로 채운다.

In [ ]:
modeling_usage = membership.merge(usage_features, on='membership_row_id', how='left')

usage_feature_cols = [c for c in usage_features.columns if c not in ['membership_row_id']]

# 시청이력 없는 고객의 수치형 행동 피처는 0으로 채운다.
for col in usage_feature_cols:
    if col == 'usage_trend_label':
        modeling_usage[col] = modeling_usage[col].fillna('no_watch')
    else:
        modeling_usage[col] = modeling_usage[col].fillna(0)

modeling_usage['has_usage_feature'] = modeling_usage['has_usage_feature'].astype(int)
modeling_usage['no_usage_feature_flag'] = (modeling_usage['has_usage_feature'] == 0).astype(int)

# 02번의 has_watch_obs와 04번의 has_usage_feature가 같은지 검산할 수 있도록 유지한다.
watch_flag_check = pd.crosstab(
    modeling_usage['has_watch_obs'],
    modeling_usage['has_usage_feature'],
    rownames=['has_watch_obs_from_02'],
    colnames=['has_usage_feature_from_04'],
)

display(watch_flag_check)
watch_flag_check.to_csv(TABLES_DIR / '04_usage_feature_watch_flag_check.csv', encoding='utf-8-sig')

print('modeling_usage shape:', modeling_usage.shape)

## 4-12. 가설형 행동 세그먼트 후보 생성

이 노트북에서는 콘텐츠 메타데이터 없이 만들 수 있는 행동 세그먼트 후보만 생성한다. 콘텐츠 기반 세그먼트는 05번 이후에 만든다.

In [ ]:
modeling_usage['stable_2screen_active'] = (
    (modeling_usage['max_screen'] == 2)
    & (modeling_usage['unique_days'] >= 3)
    & (modeling_usage['week1_ratio'] > 0)
).astype(int)

modeling_usage['discount_sensitive_risk'] = (
    (modeling_usage['is_100won'] == 1)
    & (modeling_usage['max_screen'] == 4)
    & (modeling_usage['week3_ratio'] >= 0.5)
).astype(int)

modeling_usage['promo4_late_increasing_risk'] = (
    (modeling_usage['is_100won'] == 1)
    & (modeling_usage['max_screen'] == 4)
    & (modeling_usage['w3_minus_w1_watch_time'] > 0)
).astype(int)

modeling_usage['promo2_early_routine_candidate'] = (
    (modeling_usage['is_100won'] == 1)
    & (modeling_usage['max_screen'] == 2)
    & (modeling_usage['week1_ratio'] >= 0.3)
    & (modeling_usage['unique_days'] >= 3)
).astype(int)

segment_flags = [
    'stable_2screen_active',
    'discount_sensitive_risk',
    'promo4_late_increasing_risk',
    'promo2_early_routine_candidate',
]

segment_check = []
for col in segment_flags:
    tmp = modeling_usage.groupby(col)['is_repurchase'].agg(n='count', repurchase_rate='mean').reset_index()
    tmp.insert(0, 'segment_flag', col)
    tmp = tmp.rename(columns={col: 'flag_value'})
    segment_check.append(tmp)
segment_check = pd.concat(segment_check, ignore_index=True)

display(segment_check)
segment_check.to_csv(TABLES_DIR / '04_usage_feature_behavior_segment_flag_rates.csv', index=False, encoding='utf-8-sig')

## 4-13. 주요 피처 검산표 생성

06번 유의성 검정에 들어가기 전, 주요 행동 피처들의 분포와 결측 상태를 확인한다.

In [ ]:
usage_numeric_cols = [
    'total_watch_time', 'total_sessions', 'unique_contents', 'unique_days',
    'avg_session_time', 'max_session_time', 'active_span_days', 'watch_days_ratio',
    'sessions_per_active_day', 'contents_per_active_day',
    'max_daily_watch_time', 'max_day_share', 'days_since_last_watch_to_obs_end',
    'avg_gap_between_watch_days', 'max_inactive_gap_days',
    'week1_watch_time', 'week2_watch_time', 'week3_watch_time',
    'week1_ratio', 'week2_ratio', 'week3_ratio',
    'w3_minus_w1_watch_time', 'daily_watch_slope',
]
usage_numeric_cols = [c for c in usage_numeric_cols if c in modeling_usage.columns]

feature_summary = modeling_usage[usage_numeric_cols].describe().T.reset_index().rename(columns={'index': 'feature'})
feature_summary['missing_count'] = modeling_usage[usage_numeric_cols].isna().sum().values
feature_summary['zero_count'] = (modeling_usage[usage_numeric_cols] == 0).sum().values
feature_summary['zero_rate'] = feature_summary['zero_count'] / len(modeling_usage)

feature_summary.to_csv(TABLES_DIR / '04_usage_feature_numeric_summary.csv', index=False, encoding='utf-8-sig')
display(feature_summary.head(30))

In [ ]:
usage_group_summary = modeling_usage.groupby('is_100won').agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    avg_total_watch_time=('total_watch_time', 'mean'),
    avg_unique_days=('unique_days', 'mean'),
    avg_week1_ratio=('week1_ratio', 'mean'),
    avg_week3_ratio=('week3_ratio', 'mean'),
    avg_w3_minus_w1=('w3_minus_w1_watch_time', 'mean'),
    no_watch_rate=('no_watch_obs_flag', 'mean'),
).reset_index()

usage_screen_summary = modeling_usage.groupby(['is_100won', 'max_screen'], dropna=False).agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    avg_total_watch_time=('total_watch_time', 'mean'),
    avg_unique_days=('unique_days', 'mean'),
    avg_week1_ratio=('week1_ratio', 'mean'),
    avg_week3_ratio=('week3_ratio', 'mean'),
    avg_w3_minus_w1=('w3_minus_w1_watch_time', 'mean'),
    no_watch_rate=('no_watch_obs_flag', 'mean'),
).reset_index()

usage_group_summary.to_csv(TABLES_DIR / '04_usage_feature_summary_by_100won.csv', index=False, encoding='utf-8-sig')
usage_screen_summary.to_csv(TABLES_DIR / '04_usage_feature_summary_by_100won_maxscreen.csv', index=False, encoding='utf-8-sig')

display(usage_group_summary)
display(usage_screen_summary)

## 4-14. 산출물 저장

In [ ]:
PATH_USAGE_FEATURES = INTERIM_DIR / 'user_usage_features.csv'
PATH_MODELING_USAGE = PROCESSED_DIR / 'modeling_feature_table_usage.csv'
PATH_USAGE_SUMMARY_JSON = INTERIM_DIR / 'usage_feature_summary.json'

usage_features.to_csv(PATH_USAGE_FEATURES, index=False, encoding='utf-8-sig')
modeling_usage.to_csv(PATH_MODELING_USAGE, index=False, encoding='utf-8-sig')

summary = {
    'inputs': {k: str(v) for k, v in INPUT_FILES.items()},
    'outputs': {
        'user_usage_features': str(PATH_USAGE_FEATURES),
        'modeling_feature_table_usage': str(PATH_MODELING_USAGE),
        'usage_feature_summary': str(PATH_USAGE_SUMMARY_JSON),
    },
    'rows': {
        'membership_preprocessed': int(len(membership)),
        'obs_view': int(len(obs_view)),
        'usage_features': int(len(usage_features)),
        'modeling_usage': int(len(modeling_usage)),
    },
    'feature_counts': {
        'usage_feature_columns': int(len(usage_features.columns)),
        'modeling_usage_columns': int(len(modeling_usage.columns)),
    },
    'watch_presence': {
        'has_usage_feature_rows': int(modeling_usage['has_usage_feature'].sum()),
        'no_usage_feature_rows': int(modeling_usage['no_usage_feature_flag'].sum()),
    },
    'key_rates': {
        'overall_repurchase_rate': float(modeling_usage['is_repurchase'].mean()),
        '100won_repurchase_rate': float(modeling_usage.loc[modeling_usage['is_100won'] == 1, 'is_repurchase'].mean()),
        'non_100won_repurchase_rate': float(modeling_usage.loc[modeling_usage['is_100won'] == 0, 'is_repurchase'].mean()),
    },
}

with open(PATH_USAGE_SUMMARY_JSON, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Saved:')
print(PATH_USAGE_FEATURES)
print(PATH_MODELING_USAGE)
print(PATH_USAGE_SUMMARY_JSON)

## 4-15. 04번 노트북 결론

04번 노트북의 최종 산출물은 `modeling_feature_table_usage.csv`이다. 이 파일은 02번 전처리 결과에 시청 행동 파생변수를 붙인 모델링 후보 테이블이다.

이후 단계는 다음과 같다.

1. `05_content_feature_engineering.ipynb`: 영화 메타데이터 v2를 사용해 콘텐츠 성향 피처를 생성한다.
2. `06_significance_tests.ipynb`: 04번 행동 피처와 05번 콘텐츠 피처를 포함해 모든 후보 변수와 가설을 유의성 검정한다.
3. `07_modeling_baseline.ipynb`: 06번에서 확인한 후보 변수들을 포함해 1차 모델링을 수행한다.